In [1]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import os

fake = Faker()
random.seed(42)
np.random.seed(42)

# -------------------------
# CONFIG
# -------------------------
N_CUSTOMERS = 10000
N_PRODUCTS = 100
N_ORDERS = 100000
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2026, 5, 31)

OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------
# HELPERS
# -------------------------
def random_date(start, end):
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days))


def seasonal_multiplier(date):
    """Simulate real ecommerce seasonality"""
    if date.month in [11, 12]:  # Black Friday + Christmas
        return 1.8
    if date.month == 1:  # slowdown
        return 0.7
    return 1.0


# -------------------------
# PRODUCTS
# -------------------------
categories = ["Electronics", "Fashion", "Home", "Sports", "Books", "Beauty"]

products = []
for i in range(1, N_PRODUCTS + 1):
    category = random.choice(categories)

    products.append({
        "product_id": i,
        "product_name": f"{category} Item {i}",
        "category": category,
        "unit_cost": round(random.uniform(5, 120), 2),
        "unit_price": round(random.uniform(10, 250), 2),
        "active_flag": np.random.choice([True, True, True, False], p=[0.7,0.2,0.05,0.05]),
        "launch_date": random_date(datetime(2021,1,1), START_DATE)
    })

products_df = pd.DataFrame(products)

# -------------------------
# CUSTOMERS (with dirty data)
# -------------------------
countries = ["Brazil", "Germany", "USA", "Canada", "Portugal", "Brzil", "USAA", None]

customers = []
for i in range(1, N_CUSTOMERS + 1):

    email = fake.email()
    if random.random() < 0.02:
        email = None  # missing email

    country = random.choice(countries)

    customers.append({
        "customer_id": i,
        "customer_name": fake.name(),
        "email": email,
        "country": country,
        "signup_date": random_date(datetime(2023,1,1), datetime(2026,12,31)),
        "source_channel": random.choice([
            "Google Ads", "Facebook Ads", "Instagram", "Organic", "LinkedIn", "Referral"
        ])
    })

customers_df = pd.DataFrame(customers)

# introduce duplicates (~1%)
duplicates = customers_df.sample(frac=0.01)
customers_df = pd.concat([customers_df, duplicates], ignore_index=True)

# -------------------------
# ORDERS (core table)
# -------------------------
payment_methods = ["Credit Card", "PayPal", "Pix", "Apple Pay", "Bank Transfer"]
order_statuses = ["Completed"]*85 + ["Cancelled"]*10 + ["Returned"]*5

orders = []

for i in range(1, N_ORDERS + 1):

    date = random_date(START_DATE, END_DATE)

    product_id = random.randint(1, N_PRODUCTS)
    customer_id = random.randint(1, N_CUSTOMERS)

    product = products_df.loc[products_df.product_id == product_id].iloc[0]

    quantity = np.random.choice([1,2,3,4,5,-1], p=[0.4,0.2,0.2,0.1,0.08,0.02])

    # price logic + seasonality
    base_price = product["unit_price"]
    price = base_price * seasonal_multiplier(date)

    # dirty data cases
    if random.random() < 0.002:
        product_id = None  # missing FK

    if random.random() < 0.001:
        date = datetime(2030,1,1)  # future date

    orders.append({
        "order_id": i,
        "customer_id": customer_id,
        "product_id": product_id,
        "order_date": date.date(),
        "quantity": quantity,
        "unit_price": round(price,2),
        "payment_method": random.choice(payment_methods),
        "order_status": random.choice(order_statuses)
    })

orders_df = pd.DataFrame(orders)

# duplicates (~0.5%)
orders_df = pd.concat([
    orders_df,
    orders_df.sample(frac=0.005)
], ignore_index=True)

# -------------------------
# REFUNDS
# -------------------------
refunds = []
refund_orders = orders_df.sample(frac=0.05)

for i, row in enumerate(refund_orders.itertuples()):

    refunds.append({
        "refund_id": i,
        "order_id": row.order_id,
        "refund_date": row.order_date,
        "refund_amount": row.quantity * row.unit_price,
        "refund_reason": random.choice([
            "Damaged Product",
            "Wrong Item",
            "Late Delivery",
            "Customer Changed Mind"
        ])
    })

refunds_df = pd.DataFrame(refunds)

# -------------------------
# MARKETING SPEND (seasonality heavy)
# -------------------------
channels = ["Google Ads", "Facebook Ads", "Instagram", "LinkedIn", "Organic"]

marketing = []

current = START_DATE

while current <= END_DATE:

    for channel in channels:

        base = random.uniform(200, 2000)

        # seasonality boost
        multiplier = seasonal_multiplier(current)

        if current.month in [11, 12]:
            multiplier *= 2.0

        spend = base * multiplier

        clicks = int(spend * random.uniform(1, 5))
        impressions = clicks * random.randint(5, 20)

        marketing.append({
            "date": current.date(),
            "channel": channel,
            "spend": round(spend,2),
            "clicks": clicks,
            "impressions": impressions
        })

    current += timedelta(days=1)

marketing_df = pd.DataFrame(marketing)

# -------------------------
# SESSIONS (very important for analytics engineering portfolio)
# -------------------------
sessions = []

for i in range(1, 500000):

    date = random_date(START_DATE, END_DATE)

    channel = random.choice(channels)

    converted = np.random.choice([0,1], p=[0.94, 0.06])

    sessions.append({
        "session_id": i,
        "customer_id": random.randint(1, N_CUSTOMERS),
        "session_date": date.date(),
        "channel": channel,
        "device": random.choice(["Mobile", "Desktop", "Tablet"]),
        "pageviews": random.randint(1, 20),
        "session_duration": random.randint(10, 600),
        "converted": converted
    })

sessions_df = pd.DataFrame(sessions)

# -------------------------
# SUPPORT TICKETS
# -------------------------
tickets = []

for i in range(1, 15000):

    opened = random_date(START_DATE, END_DATE)

    tickets.append({
        "ticket_id": i,
        "customer_id": random.randint(1, N_CUSTOMERS),
        "opened_date": opened.date(),
        "priority": random.choice(["Low","Medium","High","Critical"]),
        "category": random.choice(["Refund","Delivery","Payment","Account","Technical"]),
        "status": random.choice(["Open","Pending","Resolved"]),
        "resolution_days": random.randint(0, 10)
    })

tickets_df = pd.DataFrame(tickets)

# -------------------------
# SAVE FILES
# -------------------------
customers_df.to_csv(f"{OUTPUT_DIR}/customers.csv", index=False)
products_df.to_csv(f"{OUTPUT_DIR}/products.csv", index=False)
orders_df.to_csv(f"{OUTPUT_DIR}/orders.csv", index=False)
refunds_df.to_csv(f"{OUTPUT_DIR}/refunds.csv", index=False)
marketing_df.to_csv(f"{OUTPUT_DIR}/marketing_spend.csv", index=False)
sessions_df.to_csv(f"{OUTPUT_DIR}/sessions.csv", index=False)
tickets_df.to_csv(f"{OUTPUT_DIR}/support_tickets.csv", index=False)

print("✅ All datasets generated successfully in /data")

✅ All datasets generated successfully in /data
